# Vanilla RNNs and GRUs

In [1]:
import numpy as np
from numpy import random
from time import perf_counter
import tensorflow as tf

In [2]:
def sigmoid (x):
    return 1.0 / (1 + np.exp(-x))

## Forward method for vanilla RNNs and GRUs using `numpy`

- Embedding size (`emb`): 128
- Hidden state size (`h_dim`): 16

The weights `w_` and biases `b_` are initialized with dimension (`h_dim`, `emb+h_dim`) and (`h_dim`,1), and the initial hidden state `h_0` is a vector of zeros 

In [ ]:
random.seed (10)            
emb = 128                   # Embedding size
T = 256                     # Length of sequence 
h_dim = 16                  # Hidden state dimension 
h_0 = np.zeros((h_dim,1))   # Initial hidden state

# Random initialization of weights and biases 

"""
random. — use the random number generator (seeded with random.seed(10) 
    so you get the same "random" values every time you run the code, making results reproducible)
standard_normal — pick from a bell curve with mean=0 and std=1 
    as the source of those random values
"""

w1 = random.standard_normal ((h_dim, emb + h_dim))
w2 = random.standard_normal ((h_dim, emb + h_dim))
w3 = random.standard_normal ((h_dim, emb + h_dim))
b1 = random.standard_normal((h_dim, 1))
b2 = random.standard_normal((h_dim, 1))
b3 = random.standard_normal((h_dim, 1))

# Random initialization of input X
# the third dimension (1) is added to achieve the batch representation 
X = random.standard_normal((T, emb, 1))

# Define the list of weights 
weights_vanilla =[w1, b1]
weights_GRU =[w1.copy(), w2, w3, b1.copy(), b2, b3]

###  Forward method for vanilla RNNs

$$
 h^{<t>} = g(W_h \cdot h^{<t-1>} + W_x \cdot x^{<t>} + b_h)   
$$
    
$$
\hat{y}^{<t>}=g(W_{yh}h^{<t>} + b_y)
$$

where $[h^{<t-1>},x^{<t>}]$ means that $h^{<t-1>}$ and $x^{<t>}$ are concatenated together. In the next cell you have the implementation of the forward method for a vanilla RNN. 

In [4]:
def forward_V_RNN (inputs, weights):
    x, h_t = inputs

    # weights
    wh, bh = weights

    # new hidden state
    h_t = np.dot(wh, np.concatenate([h_t, x])) +bh
    h_t = sigmoid(h_t)

    # Avoid implementation of y for clarity
    y = h_t

    return y, h_t


### Forward method for GRUs

$$
\Gamma_r=\sigma{(W_r . [h^{<t-1>}, x^{<t>}]+b_r)}
$$

$$
\Gamma_u=\sigma{(W_u . [h^{<t-1>}, x^{<t>}]+b_u)}
$$

$$
c^{<t>}=\tanh{(W_h . [\Gamma_r*h^{<t-1>},x^{<t>}]+b_h)}
$$

$$
h^{<t>}=\Gamma_u*c^{<t>}+(1-\Gamma_u)*h^{<t-1>}
$$

In [5]:
def forward_GRU (inputs, weights):
    X, h_t = inputs

    # weights.
    wu, wr, wc, bu, br, bc = weights

    # Update gate
    u = np.dot(wu, np.concatenate([h_t,X])) + bu
    u = sigmoid(u)

    # Relevance gate
    r = np.dot(wr, np.concatenate([h_t,X])) + br
    r = sigmoid(r)

    # Candidate hidden state
    c = np.dot(wc, np.concatenate([r * h_t, X])) + bc
    c = np.tanh(c)

    # New hidden state
    h_t = u * c + (1-u) * h_t

    # Avoid implementation of y for clarity
    y = h_t

    return y, h_t



In [6]:
forward_GRU([X[1], h_0], weights_GRU)[0]

array([[ 9.77779014e-01],
       [-9.97986240e-01],
       [-5.19958083e-01],
       [-9.99999886e-01],
       [-9.99707004e-01],
       [-3.02197037e-04],
       [-9.58733503e-01],
       [ 2.10804828e-02],
       [ 9.77365398e-05],
       [ 9.99833090e-01],
       [ 1.63200940e-08],
       [ 8.51874303e-01],
       [ 5.21399924e-02],
       [ 2.15495959e-02],
       [ 9.99878828e-01],
       [ 9.77165472e-01]])

### Implementation fo the `scan` function
`scan` used for forward propagation in RNNs. It takes as inputs:

- `fn` : the function to be called recurrently (i.e. `forward_GRU`)
- `elems` : the list of inputs for each time step (`X`)
- `weights` : the parameters needed to compute `fn`
- `h_0` : the initial hidden state

`scan` goes through all the elements `x` in `elems`, calls the function `fn` with arguments ([`x`, `h_t`],`weights`), stores the computed hidden state `h_t` and appends the result to a list `ys`. Complete the following cell by calling `fn` with arguments ([`x`, `h_t`],`weights`).

In [7]:
def scan (fn, elems, weights, h_0):
    h_t = h_0
    ys = []

    for x in elems:
        y, h_t = fn(([x, h_t, ], weights))
        ys. append(y)

    return ys, h_t

In [ ]:
ys, h_t = scan (forward_V_RNN, X, weights_vanilla, h_0)

